# Generate correlated eQTL null pairs from TopLD

Builds the `correlated_variants_27K.csv` null control set for eQTL epistasis analysis.

**Input:** Functional eQTL pairs (Yang et al.) + TopLD per-chromosome LD tables.

**Logic:** For each functional eQTL pair, the lead eQTL position is used as an anchor.
TopLD pairs where at least one variant matches an anchor are kept as null pairs —
same genomic regions and linkage, but not necessarily functionally epistatic.

**Output:** `correlated_variants_27K.csv` with columns: epistasis_id, Uniq_ID_1, Uniq_ID_2,
chrom, distance, R2, Dprime, MAF1, MAF2.

In [ ]:
import os, sys, re
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ── Configuration ──
sys.path.insert(0, str(Path('..').resolve()))
from paper_data_config import EPISTASIS_PAPER_ROOT

TOPLD_DIR = Path(os.environ.get('TOPLD_DIR', '/tamir2/nicolaslynn/data/TopLD/data'))
PAIRS_TSV = EPISTASIS_PAPER_ROOT / 'data' / 'all_pairs_combined.tsv'
OUT_CSV = EPISTASIS_PAPER_ROOT / 'data' / 'source' / 'correlated_variants_27K.csv'

MAX_DIST_BP = 6_000
TOPLD_R2_THRESHOLD = 0.2
TOPLD_WINDOW_MB = 1_000_000

print(f'TOPLD_DIR: {TOPLD_DIR} (exists: {TOPLD_DIR.exists()})')
print(f'PAIRS_TSV: {PAIRS_TSV} (exists: {PAIRS_TSV.exists()})')
print(f'Output: {OUT_CSV}')

## TopLD helpers

In [ ]:
def topld_ld_path(chrom, topld_dir=TOPLD_DIR):
    """Return path to TopLD LD CSV for a chromosome."""
    return topld_dir / f'EUR_chr{chrom}_no_filter_{TOPLD_R2_THRESHOLD}_{TOPLD_WINDOW_MB}_LD.csv'


def topld_annot_path(chrom, topld_dir=TOPLD_DIR):
    """Return path to TopLD annotation CSV for a chromosome."""
    return topld_dir / f'EUR_chr{chrom}_no_filter_{TOPLD_R2_THRESHOLD}_{TOPLD_WINDOW_MB}_info_annotation.csv'


def load_topld_annotations(topld_dir=TOPLD_DIR):
    """Load all per-chrom TopLD annotation files into a (chrom, Uniq_ID) -> {MAF, CADD_phred} lookup."""
    lookup = {}
    for f in sorted(os.listdir(topld_dir)):
        if not f.endswith('_info_annotation.csv') or not f.startswith('EUR_chr'):
            continue
        chrom = re.search(r'chr(\d+|X)', f).group(1)
        df = pd.read_csv(topld_dir / f)
        if 'Uniq_ID' not in df.columns or 'MAF' not in df.columns:
            continue
        for _, row in df.iterrows():
            key = (str(chrom), str(row['Uniq_ID']))
            try:
                maf = float(row['MAF']) if pd.notna(row['MAF']) and str(row['MAF']).strip() not in ('.', '') else np.nan
            except (TypeError, ValueError):
                maf = np.nan
            entry = {'MAF': maf}
            if 'CADD_phred' in df.columns and pd.notna(row.get('CADD_phred')):
                try:
                    entry['CADD_phred'] = float(row['CADD_phred'])
                except (TypeError, ValueError):
                    pass
            lookup[key] = entry
    print(f'Loaded TopLD annotations: {len(lookup)} variants across {len(set(k[0] for k in lookup))} chromosomes')
    return lookup


def parse_epistasis_id(eid):
    """Parse epistasis_id -> (gene, chrom, pos1, pos2, ref1, alt1, ref2, alt2)."""
    m1, m2 = eid.split('|')
    p1 = m1.split(':')
    p2 = m2.split(':')
    return p1[0], p1[1], int(p1[2]), int(p2[2]), p1[3], p1[4], p2[3], p2[4]

print('Helpers defined.')

## 1. Load functional eQTL pairs (anchors)

In [ ]:
pairs = pd.read_csv(PAIRS_TSV, sep='\t', usecols=['source', 'epistasis_id', 'distance_bp'],
                     low_memory=False)
eqtl = pairs[pairs.source == 'yang_evqtl'].drop_duplicates('epistasis_id').copy()
print(f'Functional eQTL pairs: {len(eqtl)}')

# Extract anchor positions (pos1 = lead eQTL) per chromosome
eqtl['_parsed'] = eqtl['epistasis_id'].apply(parse_epistasis_id)
eqtl['chrom'] = eqtl['_parsed'].str[1]
eqtl['pos1'] = eqtl['_parsed'].str[2]
eqtl['pos2'] = eqtl['_parsed'].str[3]

anchors_by_chrom = eqtl.groupby('chrom')['pos1'].apply(set).to_dict()
print(f'Anchor positions across {len(anchors_by_chrom)} chromosomes: {sum(len(v) for v in anchors_by_chrom.values())} unique')

## 2. Build null pairs from TopLD

For each chromosome, scan TopLD LD tables for pairs where at least one SNP matches
a functional anchor position. Keep pairs with distance < 6 kb.

In [ ]:
matches = []

for chrom, anchor_positions in sorted(anchors_by_chrom.items(), key=lambda x: int(x[0]) if x[0].isdigit() else 99):
    ld_path = topld_ld_path(chrom)
    if not ld_path.exists():
        print(f'  chr{chrom}: LD file not found, skipping')
        continue

    for chunk in tqdm(pd.read_csv(ld_path, chunksize=1_000_000),
                      desc=f'chr{chrom}', leave=False):
        chunk = chunk.astype({'SNP1': int, 'SNP2': int})
        mask = chunk['SNP1'].isin(anchor_positions) | chunk['SNP2'].isin(anchor_positions)
        m = chunk[mask].copy()
        if not m.empty:
            m['chrom'] = chrom
            m['distance'] = (m['SNP2'] - m['SNP1']).abs()
            m = m[m.distance < MAX_DIST_BP]
            if not m.empty:
                matches.append(m)

null_df = pd.concat(matches, ignore_index=True)
print(f'\nNull pairs (distance < {MAX_DIST_BP} bp): {len(null_df)}')

## 3. Annotate with MAF and build epistasis_ids

In [ ]:
# Build epistasis_id from TopLD Uniq_IDs
null_df['epistasis_id'] = null_df.apply(
    lambda r: f"GENE:{r.chrom}:{r.Uniq_ID_1}|GENE:{r.chrom}:{r.Uniq_ID_2}", axis=1
)

# Annotate with MAF from TopLD info_annotation files
print('Loading TopLD annotations for MAF...')
topld_annot = load_topld_annotations()

null_df['MAF1'] = null_df.apply(
    lambda r: topld_annot.get((str(r.chrom), str(r.Uniq_ID_1)), {}).get('MAF', np.nan), axis=1
)
null_df['MAF2'] = null_df.apply(
    lambda r: topld_annot.get((str(r.chrom), str(r.Uniq_ID_2)), {}).get('MAF', np.nan), axis=1
)

out_cols = ['epistasis_id', 'Uniq_ID_1', 'Uniq_ID_2', 'chrom', 'distance', 'R2', 'Dprime', 'MAF1', 'MAF2']
null_df[out_cols].to_csv(OUT_CSV, index=False)
print(f'Saved {len(null_df)} null pairs to {OUT_CSV}')
null_df[out_cols].head()

## 4. Summary statistics

In [ ]:
print(f'Total null pairs: {len(null_df)}')
print(f'Chromosomes: {sorted(null_df.chrom.unique(), key=lambda x: int(x) if x.isdigit() else 99)}')
print(f'\nDistance: median={null_df.distance.median():.0f}, mean={null_df.distance.mean():.0f}, max={null_df.distance.max():.0f}')
print(f'R²: median={null_df.R2.median():.3f}, mean={null_df.R2.mean():.3f}')
print(f"D': median={null_df.Dprime.median():.3f}, mean={null_df.Dprime.mean():.3f}")
print(f'MAF1 coverage: {null_df.MAF1.notna().sum()} / {len(null_df)}')
print(f'MAF2 coverage: {null_df.MAF2.notna().sum()} / {len(null_df)}')